In [1]:
'''
Evaluate the performance of search algorithms 
Collect scores across all prompts and trials, and compute the overall statistics.
'''

import os, psutil, gc
import time 
import json
import pprint

from collections import defaultdict
import random

import numpy as np
np.set_printoptions(precision=4)
from scipy.stats import ttest_rel

In [2]:
from sal.config import Config

from datasets import load_dataset

# from core import best_of_n
from utils.load_data import load_data_prm800k
from utils import grader 
from utils import grader2
from utils import parser

In [3]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_tokenizer_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_tokenizer_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"

In [4]:
#  load data 
data_by_levels = load_data_prm800k(data_dir)


# ds_completions = load_completions(completions_dir)

# load random_seeds     
# random_seeds = np.loadtxt("random_seeds.txt").astype("int64")
# random_seeds = [int(seed) for seed in random_seeds]

1: 43
2: 90
3: 105
4: 128
5: 134


In [5]:
import signal

In [8]:
num_trials = 2
config_names_arr = [
    "bob--v11--n-8--d-40--level-4",
    "sdp--v11--n-8--bw-4--d-40--lam-10--True--dalpha-0--dbeta-1.0--ppl-True--level-4",
    "sdp--v11--n-8--bw-4--d-40--lam-10--True--dalpha-10--dbeta-1.0--ppl-True--level-4",
    "sdp--v11--n-8--bw-4--d-40--lam-10--True--dalpha-100--dbeta-1.0--ppl-True--level-4",
    "sdp--v11--n-8--bw-4--d-40--lam-10--True--dalpha-1.0--dbeta-0--ppl-True--level-4",
]

config_names_arr = [
    "bob--v11--n-8--d-40--level-4",
    "sdp--v11--n-8--bw-4--d-40--lam-10--True--dalpha-0--dbeta-1.0--ppl-True--level-4",
    "sdp--v11--n-8--bw-4--d-40--lam-10--True--dalpha-10--dbeta-1.0--ppl-True--level-4",
    "mcts--v15--n-4--d-40--nb-2--cpuct-2--level-4",
    "mcts--v55--n-4--d-40--nb-2--lam-10--dalpha-100.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
    "mcts--v35--n-4--d-40--nb-2--lam-10--dalpha-10.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
    "mcts--v36--n-4--d-40--nb-2--lam-10--dalpha-100.0--dbeta-1.0--cpuct-0-0-2--dthres-1--ppl-True--normalize-True--level-4",
    "mcts--v45--n-4--d-40--nb-2--lam-10--dalpha-100.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
]

config_names_arr = [
    "mcts--v15--n-4--d-40--nb-1--cpuct-0--level-4",
    "mcts--v15--n-4--d-40--nb-1--cpuct-0.01--level-4",
    "mcts--v15--n-4--d-40--nb-1--cpuct-0.1--level-4",
    "mcts--v15--n-4--d-40--nb-1--cpuct-1.0--level-4",
    "mcts--v15--n-4--d-40--nb-1--cpuct-2--level-4",
    "mcts--v15--n-4--d-40--nb-1--cpuct-10--level-4",
    "mcts--v15--n-4--d-40--nb-1--cpuct-100000--level-4",
]


config_names_arr = [
    "mcts--v15--n-4--d-40--nb-1--cpuct-2--level-4",
    "mcts--v15--n-4--d-40--nb-2--cpuct-2--level-4",
    "mcts--v15--n-4--d-40--nb-3--cpuct-2--level-4",
    "mcts--v15--n-4--d-40--nb-4--cpuct-2--level-4",
    "mcts--v15--n-4--d-40--nb-5--cpuct-2--level-4",
]

# config_names_arr = [
#     "bob--v11--n-8--d-40--level-4",
#     "bob--v11--n-40--d-40--level-4",
#     "mcts--v12--n-8--d-40--cpuct-2--level-4",
#     "mcts--v12--n-8--d-40--nb-5--cpuct-2--level-4",
    
#     "mcts--v51--n-8--d-40--nb-5--lam-10--dalpha-0.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v51--n-8--d-40--nb-5--lam-10--dalpha-0.1--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4",
#     "mcts--v51--n-8--d-40--nb-5--lam-10--dalpha-10.0--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4",
#     "mcts--v51--n-8--d-40--nb-5--lam-10--dalpha-100.0--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4",
#     "mcts--v51--n-8--d-40--nb-5--lam-10--dalpha-1.0--dbeta-0.0--cpuct-0-0--ppl-True--normalize-True--level-4",
# ]



# config_names_arr = [
#     "mcts--v15--n-4--d-40--nb-2--cpuct-2--level-4",
#     "mcts--v15--n-4--d-40--nb-5--cpuct-2--level-4",
#     "mcts--v55--n-4--d-40--nb-5--lam-10--dalpha-100.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v35--n-4--d-40--nb-5--lam-10--dalpha-100.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
# ]

# config_names_arr = [
#     "mcts--v15--n-4--d-40--nb-2--cpuct-2--level-4",
#     "mcts--v55--n-4--d-40--nb-2--lam-10--dalpha-0.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v55--n-4--d-40--nb-2--lam-10--dalpha-0.1--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v55--n-4--d-40--nb-2--lam-10--dalpha-1.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v55--n-4--d-40--nb-2--lam-10--dalpha-10.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v55--n-4--d-40--nb-2--lam-10--dalpha-100.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v55--n-4--d-40--nb-2--lam-10--dalpha-100000--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v55--n-4--d-40--nb-2--lam-10--dalpha-1000000000000--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v55--n-4--d-40--nb-2--lam-10--dalpha-1.0--dbeta-0.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v57--n-4--d-40--nb-2--lam-10--dalpha-1.0--dbeta-0.0--cpuct-0-0--ppl-True--normalize-True--level-4",
# ]

# config_names_arr = [
#     "mcts--v35--n-4--d-40--nb-2--lam-100--dalpha-100.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v35--n-4--d-40--nb-2--lam-10--dalpha-10.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v35--n-4--d-40--nb-2--lam-10--dalpha-100.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v35--n-4--d-40--nb-2--lam-10--dalpha-100000--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v35--n-4--d-40--nb-2--lam-10--dalpha-1.0--dbeta-0.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v37--n-4--d-40--nb-2--lam-10--dalpha-1.0--dbeta-0.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v35--n-4--d-40--nb-2--lam-1--dalpha-10.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v35--n-4--d-40--nb-2--lam-1--dalpha-100.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
# ]



nlen = len(config_names_arr)
metric_name = "naive1b"

results = []
for cidx in range(nlen):
    config_name = config_names_arr[cidx]
    print(f"\n")
    print(config_name)
    try:
        res = np.loadtxt(f"results/{config_name}/{metric_name}_{config_name}.txt")
        results.append(res[:num_trials*128])
        res_mean = np.mean(res)
        res_std = np.std(res, ddof=1)/np.sqrt(num_trials*128)
        print(f"mean = {res_mean:0.4f} (\u00B1{res_std:0.4f})")
    except Exception as e:
        
        print(f"An unexpected error occurred: {e}")
        res = None
        results.append(res)
    
        
    

for cidx1 in range(nlen):
    if results[cidx1] is None:
        pass
    for cidx2 in range(cidx1+1, nlen):
        if (results[cidx1] is None):
            pass
        try:
            t_stat, p_value = ttest_rel(results[cidx1], results[cidx2])
            print(f"\n")
            print(cidx1+1, config_names_arr[cidx1])
            print(cidx2+1, config_names_arr[cidx2])
            print(results[cidx1][:5])
            print(results[cidx2][:5])
            print(f"t_stat = {t_stat:0.3f} ({p_value:0.2f})")
            # print(f"p_value = {p_value:0.4f}")
        except Exception as e:
            # print(f"An unexpected error occurred: {e}")
            pass




mcts--v15--n-4--d-40--nb-1--cpuct-2--level-4
mean = 0.4219 (±0.0309)


mcts--v15--n-4--d-40--nb-2--cpuct-2--level-4
mean = 0.4266 (±0.0309)


mcts--v15--n-4--d-40--nb-3--cpuct-2--level-4
mean = 0.4391 (±0.0310)


mcts--v15--n-4--d-40--nb-4--cpuct-2--level-4
mean = 0.4258 (±0.0310)


mcts--v15--n-4--d-40--nb-5--cpuct-2--level-4
mean = 0.4414 (±0.0311)


1 mcts--v15--n-4--d-40--nb-1--cpuct-2--level-4
2 mcts--v15--n-4--d-40--nb-2--cpuct-2--level-4
[0. 0. 1. 1. 0.]
[0. 1. 1. 1. 0.]
t_stat = 0.000 (1.00)


1 mcts--v15--n-4--d-40--nb-1--cpuct-2--level-4
3 mcts--v15--n-4--d-40--nb-3--cpuct-2--level-4
[0. 0. 1. 1. 0.]
[0. 1. 1. 1. 0.]
t_stat = -1.363 (0.17)


1 mcts--v15--n-4--d-40--nb-1--cpuct-2--level-4
4 mcts--v15--n-4--d-40--nb-4--cpuct-2--level-4
[0. 0. 1. 1. 0.]
[0. 0. 1. 1. 0.]
t_stat = -0.156 (0.88)


1 mcts--v15--n-4--d-40--nb-1--cpuct-2--level-4
5 mcts--v15--n-4--d-40--nb-5--cpuct-2--level-4
[0. 0. 1. 1. 0.]
[0. 1. 1. 1. 0.]
t_stat = -0.729 (0.47)


2 mcts--v15--n-4--d-40--nb-2--cp